In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [41]:
pd.set_option("display.max_columns", None)

In [42]:
df = pd.read_csv('../raw_data/enriched_investments.csv', encoding='latin1', low_memory=False)
print(df.shape)

(54294, 40)


In [43]:
# drops all rows with more than 50% missing data
row_missing_pct = df.isna().mean(axis=1).mul(100)
df = df[row_missing_pct <= 50]

print(f"New shape: {df.shape}")

New shape: (49438, 40)


In [44]:
# printing missing values
missing = pd.DataFrame({
'missing_count': df.isna().sum(),
'missing_pct': df.isna().mean().mul(100).round(2),
'dtype': df.dtypes
}).sort_values('missing_pct', ascending=False)

print(missing)

                      missing_count  missing_pct    dtype
state_code                    19277        38.99   object
founded_month                 10956        22.16   object
founded_year                  10956        22.16  float64
founded_quarter               10956        22.16   object
founded_at                    10884        22.02   object
city                           6116        12.37   object
country_code                   5273        10.67   object
region                         5273        10.67   object
market                         3968         8.03   object
category_list                  3961         8.01   object
homepage_url                   3449         6.98   object
status_enriched                1314         2.66   object
status                         1314         2.66   object
round_G                           0         0.00  float64
private_equity                    0         0.00  float64
round_H                           0         0.00  float64
round_F       

In [45]:
cols_to_drop = [
'city',
'founded_quarter', # can be derived from founded_at
'founded_month', # can be derived from founded_at
'founded_year', # can be derived from founded_at
'homepage_url',
'name',
'status', # replaced by status_enriched
]

df = df.drop(columns=cols_to_drop)
print(f"New shape: {df.shape}")

New shape: (49438, 33)


In [46]:
# Check current types
print(df.dtypes)

permalink                object
category_list            object
market                   object
funding_total_usd        object
country_code             object
state_code               object
region                   object
funding_rounds          float64
founded_at               object
first_funding_at         object
last_funding_at          object
seed                    float64
venture                 float64
equity_crowdfunding     float64
undisclosed             float64
convertible_note        float64
debt_financing          float64
angel                   float64
grant                   float64
private_equity          float64
post_ipo_equity         float64
post_ipo_debt           float64
secondary_market        float64
product_crowdfunding    float64
round_A                 float64
round_B                 float64
round_C                 float64
round_D                 float64
round_E                 float64
round_F                 float64
round_G                 float64
round_H 

In [47]:
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
    df[col] = df[col].str.strip()   # remove whitespace
    df[col] = df[col].str.lower()   # standardise case

print("String columns cleaned")

String columns cleaned


In [48]:
# This fixes the column NAMES/HEADERS
df.columns = df.columns.str.strip()

In [49]:
print(df.dtypes)

permalink                object
category_list            object
market                   object
funding_total_usd        object
country_code             object
state_code               object
region                   object
funding_rounds          float64
founded_at               object
first_funding_at         object
last_funding_at          object
seed                    float64
venture                 float64
equity_crowdfunding     float64
undisclosed             float64
convertible_note        float64
debt_financing          float64
angel                   float64
grant                   float64
private_equity          float64
post_ipo_equity         float64
post_ipo_debt           float64
secondary_market        float64
product_crowdfunding    float64
round_A                 float64
round_B                 float64
round_C                 float64
round_D                 float64
round_E                 float64
round_F                 float64
round_G                 float64
round_H 

In [50]:
# Convert all date columns from object to datetime
date_cols = ['founded_at', 'first_funding_at', 'last_funding_at']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Clean funding_total_usd before converting to numeric
df['funding_total_usd'] = (df['funding_total_usd']
.str.replace(',', '', regex=False) # remove US style commas
.str.replace('-', '0', regex=False) # convert dashes to 0
.str.replace('$', '', regex=False) # remove currency symbols
)

df['funding_total_usd'] = pd.to_numeric(df['funding_total_usd'], errors='coerce')


# Verify the changes
print("\nAfter conversion:")
print(df[date_cols + ['funding_total_usd']].dtypes)
print(f"NaNs after conversion: {df['funding_total_usd'].isna().sum()}")
print(df['funding_total_usd'].describe())


After conversion:
founded_at           datetime64[ns]
first_funding_at     datetime64[ns]
last_funding_at      datetime64[ns]
funding_total_usd             int64
dtype: object
NaNs after conversion: 0
count    4.943800e+04
mean     1.316667e+07
std      1.535540e+08
min      0.000000e+00
25%      5.000000e+04
50%      1.000000e+06
75%      6.772162e+06
max      3.007950e+10
Name: funding_total_usd, dtype: float64


In [51]:
print(df.shape)

(49438, 33)


In [52]:
# How many companies founded before 2000
print(f"Companies founded before 2000: {(df['founded_at'] < '2000-01-01').sum()}")

Companies founded before 2000: 3730


In [53]:
df = df[(df['founded_at'] >= '2000-01-01') | (df['founded_at'].isna())]
print(f"New shape: {df.shape}")

New shape: (45708, 33)


In [54]:
# Checking for missing values again after updates to df were made
missing = pd.DataFrame({
'missing_count': df.isna().sum(),
'missing_pct': df.isna().mean().mul(100).round(2),
'dtype': df.dtypes
}).sort_values('missing_pct', ascending=False)

print(missing)

                      missing_count  missing_pct           dtype
state_code                    18353        40.15          object
founded_at                    10885        23.81  datetime64[ns]
country_code                   5128        11.22          object
region                         5128        11.22          object
market                         3645         7.97          object
category_list                  3638         7.96          object
status_enriched                1173         2.57          object
first_funding_at                  9         0.02  datetime64[ns]
last_funding_at                   6         0.01  datetime64[ns]
round_G                           0         0.00         float64
post_ipo_debt                     0         0.00         float64
round_F                           0         0.00         float64
round_E                           0         0.00         float64
round_D                           0         0.00         float64
round_C                  

In [55]:
df.head(10)

,permalink,category_list,market,funding_total_usd,country_code,state_code,region,funding_rounds,founded_at,first_funding_at,last_funding_at,seed,venture,equity_crowdfunding,undisclosed,convertible_note,debt_financing,angel,grant,private_equity,post_ipo_equity,post_ipo_debt,secondary_market,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H,status_enriched
0,/organization/waywire,|entertainment|politics|social media|news|,news,1750000,usa,ny,new york city,1.0,2012-06-01,2012-06-30,2012-06-30,1750000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,acquired
1,/organization/tv-communications,|games|,games,4000000,usa,ca,los angeles,2.0,NaT,2010-06-04,2010-09-23,0.0,4000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,operating
2,/organization/rock-your-paper,|publishing|education|,publishing,40000,est,NaN,tallinn,1.0,2012-10-26,2012-08-09,2012-08-09,40000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,operating
3,/organization/in-touch-network,|electronics|guides|coffee|restaurants|music|i...,electronics,1500000,gbr,NaN,london,1.0,2011-04-01,2011-04-01,2011-04-01,1500000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,closed
4,/organization/r-ranch-and-mine,|tourism|entertainment|games|,tourism,60000,usa,tx,dallas,2.0,2014-01-01,2014-08-17,2014-09-26,0.0,0.0,60000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,operating
5,/organization/club-domains,|software|,software,7000000,usa,fl,ft. lauderdale,1.0,2011-10-10,2013-05-31,2013-05-31,0.0,7000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7000000.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
6,/organization/fox-networks,|advertising|,advertising,4912393,arg,NaN,buenos aires,1.0,NaT,2007-01-16,2007-01-16,0.0,0.0,0.0,4912393.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,closed
7,/organization/0-6-com,|curated web|,curated web,2000000,NaN,NaN,NaN,1.0,2007-01-01,2008-03-19,2008-03-19,0.0,2000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2000000.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,closed
8,/organization/004-technologies,|software|,software,0,usa,il,"springfield, illinois",1.0,2010-01-01,2014-07-24,2014-07-24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,operating
9,/organization/01games-technology,|games|,games,41250,hkg,NaN,hong kong,1.0,NaT,2014-07-01,2014-07-01,41250.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,operating


In [56]:
# Categorical columns - fill with 'unknown'
cat_cols = ['state_code', 'country_code', 'region', 'market', 'category_list', 'status_enriched']
df[cat_cols] = df[cat_cols].fillna('unknown')


# Dates - fill with median
df['founded_at'] = df['founded_at'].fillna(df['founded_at'].median())
df['first_funding_at'] = df['first_funding_at'].fillna(df['first_funding_at'].median())
df['last_funding_at'] = df['last_funding_at'].fillna(df['last_funding_at'].median())

# Verify nothing is missing
print(df.isna().sum().sort_values(ascending=False))

permalink               0
angel                   0
round_H                 0
round_G                 0
round_F                 0
round_E                 0
round_D                 0
round_C                 0
round_B                 0
round_A                 0
product_crowdfunding    0
secondary_market        0
post_ipo_debt           0
post_ipo_equity         0
private_equity          0
grant                   0
debt_financing          0
category_list           0
convertible_note        0
undisclosed             0
equity_crowdfunding     0
venture                 0
seed                    0
last_funding_at         0
first_funding_at        0
founded_at              0
funding_rounds          0
region                  0
state_code              0
country_code            0
funding_total_usd       0
market                  0
status_enriched         0
dtype: int64


In [57]:
print(df.shape)

(45708, 33)


In [58]:
df = df.drop_duplicates(subset='permalink', keep='first')

In [59]:
print(df.shape)

(45706, 33)


In [60]:
sparse_cols = [
    'equity_crowdfunding', 'undisclosed', 'convertible_note',
    'product_crowdfunding', 'post_ipo_equity', 'post_ipo_debt',
    'secondary_market', 'round_F', 'round_G', 'round_H'
]

for col in sparse_cols:
    nonzero = (df[col] != 0).sum()
    nunique = df[col].nunique()
    pct_zero = (df[col] == 0).mean() * 100
    print(f"{col:25} nonzero: {nonzero:>6}  unique: {nunique:>5}  zeros: {pct_zero:.1f}%")

equity_crowdfunding       nonzero:    508  unique:   247  zeros: 98.9%
undisclosed               nonzero:    874  unique:   633  zeros: 98.1%
convertible_note          nonzero:    534  unique:   288  zeros: 98.8%
product_crowdfunding      nonzero:    208  unique:   172  zeros: 99.5%
post_ipo_equity           nonzero:    245  unique:   195  zeros: 99.5%
post_ipo_debt             nonzero:     49  unique:    40  zeros: 99.9%
secondary_market          nonzero:     16  unique:    17  zeros: 100.0%
round_F                   nonzero:    139  unique:    92  zeros: 99.7%
round_G                   nonzero:     26  unique:    25  zeros: 99.9%
round_H                   nonzero:      4  unique:     5  zeros: 100.0%


In [61]:
df = df.drop(columns=sparse_cols)
print(f"Dropped {len(sparse_cols)} sparse columns. New shape: {df.shape}")

Dropped 10 sparse columns. New shape: (45706, 23)


In [62]:
funding_cols = ['seed', 'venture', 'angel', 'debt_financing', 'private_equity',
                'round_A', 'round_B', 'round_C', 'round_D', 'round_E',
                'funding_total_usd']

for col in funding_cols:
    p99 = df[col].quantile(0.99)
    above = (df[col] > p99).sum()
    max_val = df[col].max()
    print(f"{col:25} p99: {p99:>15,.0f}  above: {above:>4}  max: {max_val:>15,.0f}")

seed                      p99:       3,100,000  above:  451  max:     130,000,000
venture                   p99:      99,581,386  above:  458  max:   2,351,000,000
angel                     p99:       1,600,000  above:  453  max:      63,590,263
debt_financing            p99:      10,000,000  above:  455  max:   3,200,000,000
private_equity            p99:      25,000,000  above:  449  max:   2,600,000,000
round_A                   p99:      20,000,000  above:  394  max:     319,000,000
round_B                   p99:      28,000,000  above:  451  max:     542,000,000
round_C                   p99:      31,000,000  above:  453  max:     490,000,000
round_D                   p99:      21,597,511  above:  458  max:   1,200,000,000
round_E                   p99:               0  above:  422  max:     400,000,000
funding_total_usd         p99:     145,373,698  above:  458  max:   5,800,000,000


In [63]:
funding_cols = ['seed', 'venture', 'angel', 'debt_financing', 'private_equity',
                'round_A', 'round_B', 'round_C', 'round_D', 'round_E',
                'funding_total_usd']

# Only clip columns that still exist after dropping sparse
funding_cols = [c for c in funding_cols if c in df.columns]

for col in funding_cols:
    cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap)
    print(f"{col:25} capped at {cap:>15,.0f}")

seed                      capped at       3,100,000
venture                   capped at      99,581,386
angel                     capped at       1,600,000
debt_financing            capped at      10,000,000
private_equity            capped at      25,000,000
round_A                   capped at      20,000,000
round_B                   capped at      28,000,000
round_C                   capped at      31,000,000
round_D                   capped at      21,597,511
round_E                   capped at               0
funding_total_usd         capped at     145,373,698


In [64]:
before = df.shape[0]
df = df[df['funding_rounds'] <= 10]
print(f"Dropped {before - df.shape[0]} rows with >10 funding rounds. New shape: {df.shape}")

Dropped 57 rows with >10 funding rounds. New shape: (45649, 23)


In [65]:
# Check enriched status distribution
print("Status enriched distribution:")
print(df['status_enriched'].value_counts())
print(f"\nClass balance if target = acquired+operating vs closed:")
success = df['status_enriched'].isin(['acquired', 'operating']).sum()
failure = (df['status_enriched'] == 'closed').sum()
unknown = (df['status_enriched'] == 'unknown').sum()
print(f"  Success: {success} ({100*success/len(df):.1f}%)")
print(f"  Failure: {failure} ({100*failure/len(df):.1f}%)")
print(f"  Unknown: {unknown} ({100*unknown/len(df):.1f}%) - will be dropped")

Status enriched distribution:
status_enriched
operating    30046
closed       11338
acquired      3092
unknown       1173
Name: count, dtype: int64

Class balance if target = acquired+operating vs closed:
  Success: 33138 (72.6%)
  Failure: 11338 (24.8%)
  Unknown: 1173 (2.6%) - will be dropped


In [66]:
funded_before = df[df['first_funding_at'] < df['founded_at']]
print(f"Funded before founded: {len(funded_before)} rows")
print(f"\nSample:")
print(funded_before[['permalink', 'founded_at', 'first_funding_at', 'funding_total_usd']].head(10))
print(f"\nHow far off (days):")
diff = (funded_before['founded_at'] - funded_before['first_funding_at']).dt.days
print(diff.describe())

Funded before founded: 6022 rows

Sample:
                          permalink founded_at first_funding_at  \
2     /organization/rock-your-paper 2012-10-26       2012-08-09   
6        /organization/fox-networks 2010-02-01       2007-01-16   
20       /organization/1000memories 2010-07-01       2010-01-01   
24           /organization/100du-tv 2010-02-01       2008-01-07   
25           /organization/100e-com 2010-02-01       2006-01-01   
52            /organization/139shop 2010-02-01       2007-02-01   
54          /organization/140-proof 2010-01-11       2009-07-01   
56  /organization/phoneuser-network 2010-02-01       2008-02-01   
68          /organization/1calendar 2009-01-19       2008-04-04   
73          /organization/1daylater 2009-08-26       2009-05-01   

    funding_total_usd  
2             40000.0  
6           4912393.0  
20          2535000.0  
24          3000000.0  
25          4500000.0  
52                0.0  
54          5500000.0  
56          6204822.0  
68  

In [67]:
diff_days = (df.loc[df['first_funding_at'] < df['founded_at'], 'founded_at'] -
             df.loc[df['first_funding_at'] < df['founded_at'], 'first_funding_at']).dt.days

print(f"Number of affected rows: {len(diff_days)}")
print(f"\nDifference in days (founded_at - first_funding_at):")
print(diff_days.describe())
print(f"\nIn years:")
print((diff_days / 365.25).describe())

Number of affected rows: 6022

Difference in days (founded_at - first_funding_at):
count     6022.000000
mean       645.455164
std       1018.393680
min          1.000000
25%         90.000000
50%        306.000000
75%        929.750000
max      32295.000000
dtype: float64

In years:
count    6022.000000
mean        1.767160
std         2.788210
min         0.002738
25%         0.246407
50%         0.837782
75%         2.545517
max        88.418891
dtype: float64


In [68]:
# Check how many we'd lose at different cutoffs
diff_days = (df['founded_at'] - df['first_funding_at']).dt.days
funded_before = diff_days > 0  # founded_at is AFTER first_funding_at

for years in [1, 2, 3, 5]:
    threshold = years * 365
    to_drop = (diff_days > threshold).sum()
    print(f"Gap > {years} year(s): {to_drop} rows")

Gap > 1 year(s): 2718 rows
Gap > 2 year(s): 1879 rows
Gap > 3 year(s): 1286 rows
Gap > 5 year(s): 354 rows


In [69]:
diff_days = (df['founded_at'] - df['first_funding_at']).dt.days
garbage_dates = diff_days > (3 * 365)
print(f"Dropping {garbage_dates.sum()} rows with funded > 3 years before founded")
df = df[~garbage_dates]
print(f"New shape: {df.shape}")

Dropping 1286 rows with funded > 3 years before founded
New shape: (44363, 23)


In [70]:
mask = (df['funding_total_usd'] == 0) & (df['funding_rounds'] > 0)
print(f"Total 0-funding rows: {mask.sum()}")
print(f"\nStatus distribution of 0-funding companies:")
print(df.loc[mask, 'status_enriched'].value_counts())

Total 0-funding rows: 7712

Status distribution of 0-funding companies:
status_enriched
operating    5315
closed       1920
acquired      322
unknown       155
Name: count, dtype: int64


In [71]:
mask = (df['funding_total_usd'] == 0) & (df['funding_rounds'] > 0) & (df['status_enriched'] == 'acquired')

funding_cols = ['seed', 'venture', 'angel', 'grant', 'debt_financing',
                'private_equity', 'round_A', 'round_B', 'round_C',
                'round_D', 'round_E']

funding_cols = [c for c in funding_cols if c in df.columns]

print(f"Total rows: {mask.sum()}")
print(f"\nSum across all financial columns:")
print(df.loc[mask, funding_cols].sum())
print(f"\nAny row with ANY nonzero funding column:")
has_any = (df.loc[mask, funding_cols] != 0).any(axis=1).sum()
print(f"{has_any} rows")

Total rows: 322

Sum across all financial columns:
seed              0.0
venture           0.0
angel             0.0
grant             0.0
debt_financing    0.0
private_equity    0.0
round_A           0.0
round_B           0.0
round_C           0.0
round_D           0.0
round_E           0.0
dtype: float64

Any row with ANY nonzero funding column:
0 rows


In [72]:
mask = (df['funding_total_usd'] == 0) & (df['funding_rounds'] > 0)
print(f"Rows to drop: {mask.sum()}")
print(f"\nStatus of rows being dropped:")
print(df.loc[mask, 'status_enriched'].value_counts())

print(f"\nAfter drop:")
df_after = df[~mask]
print(f"Shape: {df_after.shape}")
print(f"\nClass balance:")
print(df_after['status_enriched'].value_counts())
success = df_after['status_enriched'].isin(['acquired', 'operating']).sum()
failure = (df_after['status_enriched'] == 'closed').sum()
total = success + failure
print(f"\nSuccess: {success} ({100*success/total:.1f}%)")
print(f"Failure: {failure} ({100*failure/total:.1f}%)")

Rows to drop: 7712

Status of rows being dropped:
status_enriched
operating    5315
closed       1920
acquired      322
unknown       155
Name: count, dtype: int64

After drop:
Shape: (36651, 23)

Class balance:
status_enriched
operating    24047
closed        9111
acquired      2497
unknown        996
Name: count, dtype: int64

Success: 26544 (74.4%)
Failure: 9111 (25.6%)


In [73]:
mask = (df['funding_total_usd'] == 0) & (df['funding_rounds'] > 0) & (df['status_enriched'] == 'acquired')
print(f"Acquired with 0 funding: {mask.sum()}")
print(f"\nSample:")
print(df.loc[mask, ['permalink', 'funding_total_usd', 'funding_rounds', 'seed', 'venture',
                     'angel', 'round_A', 'round_B', 'market', 'country_code']].head(15).to_string())

Acquired with 0 funding: 322

Sample:
                                  permalink  funding_total_usd  funding_rounds  seed  venture  angel  round_A  round_B                   market country_code
50                  /organization/12society                0.0             1.0   0.0      0.0    0.0      0.0      0.0               e-commerce          usa
134                    /organization/29west                0.0             1.0   0.0      0.0    0.0      0.0      0.0                  unknown      unknown
338                       /organization/5by                0.0             1.0   0.0      0.0    0.0      0.0      0.0              photography      unknown
446                  /organization/9cookies                0.0             1.0   0.0      0.0    0.0      0.0      0.0              restaurants      unknown
616            /organization/academic-earth                0.0             1.0   0.0      0.0    0.0      0.0      0.0                    video          usa
657                 

In [74]:
mask = (df['funding_total_usd'] == 0) & (df['funding_rounds'] > 0)
zero_funding = df[mask]
print(f"Rows with 0 funding but has rounds: {len(zero_funding)}")
print(f"\nDo they have values in individual funding columns?")
funding_cols_check = ['seed', 'venture', 'angel', 'grant', 'debt_financing',
                      'private_equity', 'round_A', 'round_B']
print(zero_funding[funding_cols_check].sum())
print(f"\nSample:")
print(zero_funding[['permalink', 'funding_total_usd', 'funding_rounds', 'seed', 'venture']].head(10))

Rows with 0 funding but has rounds: 7712

Do they have values in individual funding columns?
seed              0.0
venture           0.0
angel             0.0
grant             0.0
debt_financing    0.0
private_equity    0.0
round_A           0.0
round_B           0.0
dtype: float64

Sample:
                          permalink  funding_total_usd  funding_rounds  seed  \
8    /organization/004-technologies                0.0             1.0   0.0   
11            /organization/1-4-all                0.0             1.0   0.0   
14   /organization/1-618-technology                0.0             1.0   0.0   
19  /organization/1000jobboersen-de                0.0             1.0   0.0   
29          /organization/10â°north                0.0             1.0   0.0   
33              /organization/10six                0.0             1.0   0.0   
36  /organization/115-network-disks                0.0             1.0   0.0   
39          /organization/fitfrnd-2                0.0             

In [75]:
# Save cleaned enriched data
df.to_csv('../raw_data/cleaned_enriched_data.csv', index=False)
print(f"Saved cleaned enriched data: {df.shape}")
print(f"\nFinal status distribution:")
print(df['status_enriched'].value_counts())

Saved cleaned enriched data: (44363, 23)

Final status distribution:
status_enriched
operating    29362
closed       11031
acquired      2819
unknown       1151
Name: count, dtype: int64
